# HydroSeason Quickstart

Install with `pip install hydroseason`, then run this notebook from anywhere.

Minimal notebook workflow:
1. Load monthly data
2. Run hydrological delineation
3. Inspect the summary card (regime, SI, key stats)
4. Explore interactive season plots
5. Export a self-contained HTML report

In [ ]:
from pathlib import Path

from hydroseason import read_rainfall

data_path = Path("../data/DATASET.csv")
df = read_rainfall(data_path, source="csv")
df.head()

,Unnamed: 0,Date,Year,Month,Rainfall_mm,Rain_Smoothed,Significant,SeasonType,SeasonShift,Hydro_Year,Dry_season_rain_count,Rain_dry_season_mm,Rain_wet_season_mm,zero_flow_months_count,Dry_month_count,Drought_Category,Year_Class_SPI
0,0,1986-12-01,1986,12,26.5,16.0,True,Wet,True,1987,0,0.0,642.5,4,4,Regular,Regular
1,1,1987-01-01,1987,1,214.5,120.5,True,Wet,False,1987,0,0.0,642.5,4,4,Regular,Regular
2,2,1987-02-01,1987,2,220.5,217.5,True,Wet,False,1987,0,0.0,642.5,4,4,Regular,Regular
3,3,1987-03-01,1987,3,149.5,185.0,True,Wet,False,1987,0,0.0,642.5,4,4,Regular,Regular
4,4,1987-04-01,1987,4,2.5,76.0,True,Wet,False,1987,0,0.0,642.5,4,4,Regular,Regular


In [ ]:
from hydroseason import classify_rainfall
from hydroseason.report import display_summary

artifacts = classify_rainfall(df)
result = artifacts.result

display_summary(artifacts)

### What `classify_rainfall` returns

`artifacts` is a `PipelineArtifacts` bundle with five fields:

- **`result`** — your input rows plus `SeasonType`, `Hydro_Year`, and metric columns (this is the main output).
- **`fixed_monthly`** — the 12-row monthly climatology and baseline Wet/Dry label per calendar month.
- **`wet_boundaries`** — per-hydro-year wet-season start/end boundaries (`None` for non-seasonal regimes).
- **`seasonality`** — STL strength, Walsh-Lawler SI, and the detected regime.
- **`diagnostics`** — a record of every algorithm decision (also written to the `.HydroSeason.json` sidecar).

If validation fails (for example a missing `Rainfall_mm` column or a data gap longer than `max_consecutive_imputation_gap` months), the call raises with a message describing the problem.

In [ ]:
from hydroseason.plot import plot_imputation_overview, show

print(f"Data confidence: {artifacts.diagnostics.data_confidence}")
print(f"Imputed months: {artifacts.diagnostics.n_imputed}")
show(plot_imputation_overview(result))

In [15]:
from hydroseason.plot import plot_season_timeline, show

# Interactive bar chart — coloured by SeasonType, hydro-year boundaries, range slider
# show() enables scroll-to-zoom and responsive sizing
show(plot_season_timeline(result))

In [16]:
from hydroseason.plot import plot_monthly_climatology, show

# Mean monthly rainfall coloured by baseline season assignment, with std error bars
show(plot_monthly_climatology(result, artifacts.fixed_monthly))

In [17]:
from hydroseason.plot import plot_annual_metrics, show

# Stacked wet/dry totals per hydrological year + wet month count
show(plot_annual_metrics(result))

In [18]:
from hydroseason.plot import plot_dashboard, show

# Composite dashboard: timeline + climatology + annual totals in one figure
show(plot_dashboard(artifacts))

In [19]:
from hydroseason.report import generate_html_report

# Export a self-contained HTML report (open in any browser — no Python needed)
report_path = generate_html_report(artifacts, "hydroseason_report.html")
print(f"Report written to: {report_path.resolve()}")

Report written to: D:\RLH\5.6\repos\hydroseason\notebooks\hydroseason_report.html


In [10]:
from hydroseason.report import export_bundle

# Export full bundle: offline HTML report + CSV + JSON.
# Add export_png=True when static PNG files are needed and Kaleido/Chrome are available.
bundle_path = export_bundle(artifacts, "hydroseason_export")
print(f"Bundle written to: {bundle_path}")
print("Contents:")
for p in sorted(bundle_path.rglob("*")):
    print(f"  {p.relative_to(bundle_path)}")

Bundle written to: D:\RLH\5.6\repos\hydroseason\notebooks\hydroseason_export
Contents:
  data
  data\diagnostics.json
  data\metrics_annual.csv
  data\results_monthly.csv
  report.html


C:\Users\00101125\AppData\Local\Temp\ipykernel_39344\2529704473.py:4: UserWarning: kaleido is not installed — PNG figure export skipped. Install with: pip install kaleido
  bundle_path = export_bundle(artifacts, "hydroseason_export")
